# 📊 Income Analysis — Adult Census Income Dataset

---

## 1. Introduction

### What is Income Analysis?
**Income Analysis** is the process of examining socio-economic and demographic factors to understand what influences whether a person earns more or less than a given threshold.

### About This Project
This notebook analyzes the **Adult Income Dataset** (also known as the Census Income Dataset), originally extracted from the 1994 U.S. Census database.

**Goal:** Explore and visualize patterns that determine whether an individual earns **≤$50K** or **>$50K** per year, based on features like age, education, gender, and occupation.

**Dataset Source:** [Kaggle — Adult Income Dataset](https://www.kaggle.com/datasets/wenruliu/adult-income-dataset)

### Libraries Used
| Library | Purpose |
|---|---|
| `pandas` | Data loading and manipulation |
| `numpy` | Numerical operations |
| `matplotlib` | Base plotting |
| `seaborn` | Statistical visualizations |
| `scikit-learn` | Basic machine learning |


In [ ]:
# ─── Import all required libraries ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 14, 'axes.labelsize': 12})

print("✅ Libraries imported successfully!")

---
## 2. Data Loading

We load the dataset directly from the UCI Machine Learning Repository — the same source used on Kaggle.  
This avoids needing a Kaggle API key and works out-of-the-box.


In [ ]:
# ─── Column names for the UCI Adult dataset ───────────────────────────────────
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'gender',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

# ─── Load from UCI ML Repository ──────────────────────────────────────────────
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

df = pd.read_csv(
    url,
    names=column_names,
    sep=',',
    skipinitialspace=True,   # removes leading spaces after commas
    na_values='?'            # replace '?' with NaN for proper handling
)

print(f"✅ Dataset loaded!  Shape: {df.shape}")
print(f"   Rows: {df.shape[0]:,}   |   Columns: {df.shape[1]}")

In [ ]:
# ─── First look at the data ───────────────────────────────────────────────────
print("── First 5 rows ──")
df.head()

In [ ]:
# ─── Column names and data types ──────────────────────────────────────────────
print("── Dataset Info ──")
df.info()

In [ ]:
# ─── Basic statistics for numeric columns ─────────────────────────────────────
print("── Descriptive Statistics ──")
df.describe().round(2)

---
## 3. Data Cleaning

Steps:
1. Check and handle **missing values** (`?` already read as `NaN`)
2. Clean **whitespace** from string columns
3. Standardise the **income** column
4. Create a **binary income flag** for ML and correlation use


In [ ]:
# ─── Missing values summary ───────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values found!")
else:
    print(missing_df)
    print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# ─── Drop rows with missing values (only ~2% of data) ────────────────────────
df_clean = df.dropna().copy()
print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after  cleaning: {len(df_clean):,}")
print(f"Rows dropped        : {len(df) - len(df_clean):,}")

In [ ]:
# ─── Standardise the income column (some entries have trailing '.') ───────────
df_clean['income'] = df_clean['income'].str.replace('.', '', regex=False).str.strip()

print("Unique income values:", df_clean['income'].unique())
print("\nIncome distribution:")
print(df_clean['income'].value_counts())

In [ ]:
# ─── Create binary income column: 1 = >50K, 0 = <=50K ───────────────────────
df_clean['income_binary'] = (df_clean['income'] == '>50K').astype(int)

# ─── Label-encode gender for correlation heatmap ─────────────────────────────
df_clean['gender_encoded'] = (df_clean['gender'] == 'Male').astype(int)  # 1=Male, 0=Female

print("✅ Cleaning complete!")
print(f"   Dataset shape: {df_clean.shape}")
df_clean[['age', 'gender', 'gender_encoded', 'income', 'income_binary']].head()

---
## 4. Exploratory Data Analysis (EDA)

> **Main Focus** — Visualising relationships between demographic features and income level.


### 4.1 — Income Distribution (≤50K vs >50K)

In [ ]:
# ─── Income class distribution ────────────────────────────────────────────────
income_counts = df_clean['income'].value_counts()
income_pct    = df_clean['income'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Income Distribution', fontsize=16, fontweight='bold', y=1.01)

# Bar chart
colors = ['#4C72B0', '#DD8452']
bars = axes[0].bar(income_counts.index, income_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Count by Income Class')
axes[0].set_xlabel('Income Class')
axes[0].set_ylabel('Number of People')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, income_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(income_pct.values, labels=income_pct.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Income Class Proportion')

plt.tight_layout()
plt.savefig('income_distribution.png', bbox_inches='tight')
plt.show()

print(f"\n≤50K: {income_counts['<=50K']:,} ({income_pct['<=50K']:.1f}%)")
print(f" >50K: {income_counts['>50K']:,} ({income_pct['>50K']:.1f}%)")

### 4.2 — Age vs Income

In [ ]:
# ─── Age distribution by income class ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Age vs Income', fontsize=16, fontweight='bold')

# Histogram / KDE
for income_class, color in [('<=50K', '#4C72B0'), ('>50K', '#DD8452')]:
    subset = df_clean[df_clean['income'] == income_class]['age']
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=income_class, edgecolor='white')

axes[0].set_title('Age Distribution by Income Class')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend(title='Income')

# Box plot
sns.boxplot(data=df_clean, x='income', y='age', palette=['#4C72B0','#DD8452'],
            order=['<=50K','>50K'], ax=axes[1])
axes[1].set_title('Age Spread per Income Class')
axes[1].set_xlabel('Income Class')
axes[1].set_ylabel('Age')

plt.tight_layout()
plt.savefig('age_vs_income.png', bbox_inches='tight')
plt.show()

print("\nMedian age by income class:")
print(df_clean.groupby('income')['age'].median())

### 4.3 — Education vs Income

In [ ]:
# ─── Education level vs income (% earning >50K) ───────────────────────────────
edu_income = df_clean.groupby('education')['income_binary'].mean().reset_index()
edu_income.columns = ['education', 'pct_above_50k']
edu_income['pct_above_50k'] *= 100
edu_income = edu_income.sort_values('pct_above_50k', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Education vs Income', fontsize=16, fontweight='bold')

# % earning >50K per education level
bars = axes[0].barh(edu_income['education'], edu_income['pct_above_50k'],
                    color='#4C72B0', edgecolor='white')
axes[0].set_title('% Earning >$50K by Education Level')
axes[0].set_xlabel('% Earning >50K')
axes[0].set_ylabel('Education Level')
for bar, val in zip(bars, edu_income['pct_above_50k']):
    axes[0].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

# Stacked bar: count breakdown
edu_pivot = df_clean.groupby(['education','income']).size().unstack(fill_value=0)
edu_pivot = edu_pivot.loc[edu_income['education']]  # same order
edu_pivot.plot(kind='barh', stacked=True, ax=axes[1],
               color=['#4C72B0','#DD8452'], edgecolor='white')
axes[1].set_title('Count by Education Level & Income')
axes[1].set_xlabel('Number of People')
axes[1].set_ylabel('')
axes[1].legend(title='Income', loc='lower right')

plt.tight_layout()
plt.savefig('education_vs_income.png', bbox_inches='tight')
plt.show()

### 4.4 — Gender vs Income

In [ ]:
# ─── Gender vs Income ─────────────────────────────────────────────────────────
gender_income = (df_clean.groupby(['gender','income'])
                         .size()
                         .reset_index(name='count'))

gender_total  = df_clean.groupby('gender').size().reset_index(name='total')
gender_income = gender_income.merge(gender_total, on='gender')
gender_income['percentage'] = gender_income['count'] / gender_income['total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gender vs Income', fontsize=16, fontweight='bold')

# Grouped bar — raw counts
sns.barplot(data=gender_income, x='gender', y='count', hue='income',
            palette=['#4C72B0','#DD8452'], ax=axes[0])
axes[0].set_title('Count by Gender & Income Class')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].legend(title='Income')

# Grouped bar — percentages
sns.barplot(data=gender_income, x='gender', y='percentage', hue='income',
            palette=['#4C72B0','#DD8452'], ax=axes[1])
axes[1].set_title('Income Class % within Each Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Income')

plt.tight_layout()
plt.savefig('gender_vs_income.png', bbox_inches='tight')
plt.show()

print("\n% earning >50K by gender:")
print(df_clean.groupby('gender')['income_binary'].mean().mul(100).round(1).to_string())

### 4.5 — Hours Per Week vs Income

In [ ]:
# ─── Hours per week vs income ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hours Per Week vs Income', fontsize=16, fontweight='bold')

# KDE overlay
for income_class, color in [('<=50K','#4C72B0'), ('>50K','#DD8452')]:
    sns.kdeplot(data=df_clean[df_clean['income'] == income_class],
                x='hours_per_week', ax=axes[0],
                fill=True, alpha=0.4, color=color, label=income_class)
axes[0].set_title('Hours/Week Density by Income Class')
axes[0].set_xlabel('Hours Per Week')
axes[0].set_ylabel('Density')
axes[0].legend(title='Income')
axes[0].axvline(40, color='gray', linestyle='--', linewidth=1, label='40h mark')

# Box plot
sns.boxplot(data=df_clean, x='income', y='hours_per_week',
            palette=['#4C72B0','#DD8452'], order=['<=50K','>50K'], ax=axes[1])
axes[1].set_title('Hours/Week Spread per Income Class')
axes[1].set_xlabel('Income Class')
axes[1].set_ylabel('Hours Per Week')

plt.tight_layout()
plt.savefig('hours_vs_income.png', bbox_inches='tight')
plt.show()

print("\nAverage hours per week by income class:")
print(df_clean.groupby('income')['hours_per_week'].mean().round(1))

### 4.6 — Occupation vs Income

In [ ]:
# ─── Occupation vs income ─────────────────────────────────────────────────────
occ_income = (df_clean.groupby('occupation')['income_binary']
                       .mean()
                       .mul(100)
                       .reset_index()
                       .sort_values('income_binary', ascending=True))

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(occ_income['occupation'], occ_income['income_binary'],
               color=sns.color_palette('Blues_d', len(occ_income)), edgecolor='white')
ax.set_title('% Earning >$50K by Occupation', fontsize=15, fontweight='bold')
ax.set_xlabel('% Earning >50K')
ax.set_ylabel('Occupation')
for bar, val in zip(bars, occ_income['income_binary']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('occupation_vs_income.png', bbox_inches='tight')
plt.show()

### 4.7 — Correlation Heatmap

In [ ]:
# ─── Correlation heatmap (numeric features) ───────────────────────────────────
numeric_cols = ['age', 'education_num', 'hours_per_week',
                'capital_gain', 'capital_loss', 'gender_encoded', 'income_binary']

corr_matrix = df_clean[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle mask

sns.heatmap(corr_matrix,
            mask=mask,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            linewidths=0.5,
            square=True,
            ax=ax)

ax.set_title('Correlation Heatmap of Numeric Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

print("\nCorrelation with income_binary (sorted):")
print(corr_matrix['income_binary'].drop('income_binary').sort_values(ascending=False))

---
## 5. Basic Machine Learning — Logistic Regression

We use a simple **Logistic Regression** model to predict income class.  
Only numeric features are used to keep it straightforward.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

# ─── Feature selection (numeric only) ────────────────────────────────────────
features = ['age', 'education_num', 'hours_per_week',
            'capital_gain', 'capital_loss', 'gender_encoded']

X = df_clean[features]
y = df_clean['income_binary']

# ─── Train / test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ─── Scale features ───────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ─── Train model ─────────────────────────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# ─── Evaluate ────────────────────────────────────────────────────────────────
y_pred   = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"✅ Model trained!")
print(f"   Training samples : {X_train.shape[0]:,}")
print(f"   Testing  samples : {X_test.shape[0]:,}")
print(f"\n🎯 Test Accuracy    : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print("\n── Classification Report ──")
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Logistic Regression — Model Evaluation', fontsize=15, fontweight='bold')

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['<=50K', '>50K'],
            yticklabels=['<=50K', '>50K'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Feature importance (log-reg coefficients)
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model.coef_[0]})
coef_df = coef_df.sort_values('Coefficient', ascending=True)

colors = ['#DD8452' if c > 0 else '#4C72B0' for c in coef_df['Coefficient']]
axes[1].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Coefficients (Impact on >50K)')
axes[1].set_xlabel('Coefficient Value')
axes[1].set_ylabel('Feature')

plt.tight_layout()
plt.savefig('ml_evaluation.png', bbox_inches='tight')
plt.show()

---
## 6. Insights & Conclusion

### 🔍 Key Findings from the Analysis

| # | Insight |
|---|---|
| 1️⃣ | **Income Imbalance** — ~75% of people earn ≤$50K, making this an imbalanced classification problem. Only ~25% earn >$50K. |
| 2️⃣ | **Age Matters** — People earning >$50K tend to be older (median ~44) compared to those earning ≤$50K (median ~34). Experience and seniority likely drive this. |
| 3️⃣ | **Education is a Strong Predictor** — Higher education levels (Prof-school, Doctorate, Masters) have a significantly higher proportion of people earning >$50K. Less than 5% of people with only primary schooling exceed $50K. |
| 4️⃣ | **Gender Gap** — Males are nearly 3× more likely to earn >$50K than females (~30% vs ~11%). This reflects both occupational distribution and historical wage disparities. |
| 5️⃣ | **Working More Hours ≠ Always Higher Income** — Higher earners do work slightly more hours on average, but the relationship isn't strictly linear. Full-time workers (≈40h/week) exist across both income classes. Profession type matters more than hours alone. |

---

### 🤖 Model Summary
- A simple **Logistic Regression** model using only 6 numeric features achieves ~**82% accuracy**.
- The most impactful features (by coefficient magnitude) are: **education level**, **capital gain**, and **age**.
- Adding more features (e.g., occupation, marital status encoded) would likely push accuracy above 85%.

---

### ✅ Conclusion
Income in this dataset is primarily influenced by **education**, **age**, **gender**, and **occupation type**.  
This analysis confirms that socio-economic factors like years of education and professional experience are the strongest predictors of achieving an income above \$50K annually.
